For 1 item/store I want to do a search over possible architectures


In [74]:
# Add project root to path so we can import from data_manipulation and model
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [75]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial

# Imports from data_manipulation and model
from data_manipulation.data_split import create_dataloader, DemandDataset
from model.functions import pinball_loss, rmse, train, get_test_loss

In [76]:
class simple_net(nn.Module):
    def __init__(self, layers:list[int]):
        super().__init__()
        
        self.net = nn.Sequential()

        # Add layers
        for i, layer in enumerate(layers[:-2]):
            self.net.add_module(f"fc{i}", nn.Linear(layer, layers[i+1]))
            if i < len(layers) - 1:
                self.net.add_module(f"relu{i}", nn.ReLU())

        # Add last layer
        self.net.add_module(f"fc{len(layers)-1}", nn.Linear(layers[-2], layers[-1]))
        

    def forward(self, x):
        return self.net(x)

In [77]:
all_specs = [
    "7_day_rolling_ema",	
    "7_day_rolling_mean",	
    "30_day_rolling_ema",	
    "diff_180_day",	
    "diff_90_day",	
    "30_day_rolling_mean",	
    "diff_30_day",	
    "7_day_rolling_min",	
    "7_day_lag",	
    "30_day_rolling_min",	
    "14_day_lag",	
    "diff_365_day",	
]

In [78]:
h_cost = 1
l_cost = 3

num_epochs = 200

In [ ]:
in_features = 12
out_features = 1

layer_combos =[
    [in_features, 30, out_features],
    [in_features, 30, 30, out_features],
    [in_features, 30, 30, 30, out_features],
    [in_features, 30, 30, 30, 30, out_features],
    [in_features, 30, 30, 30, 30, 30, out_features],
    [in_features, 40, out_features],
    [in_features, 40, 40, out_features],
    [in_features, 40, 40, 40, out_features],
    [in_features, 40, 40, 40, 40, out_features],
    [in_features, 40, 40, 40, 40, 40, out_features],
    [in_features, 50, out_features],
    [in_features, 50, 50, out_features],
    [in_features, 50, 50, 50, 50, out_features],
    [in_features, 50, 50, 50, 50, 50, out_features],
    [in_features, 50, 50, 50, 50, 50, 50, out_features],
    [in_features, 60, out_features],
    [in_features, 60, 60, out_features],
    [in_features, 60, 60, 60, out_features],
    [in_features, 60, 60, 60, 60, out_features], 
    [in_features, 60, 60, 60, 60, 60, out_features],
]

In [80]:
from collections import defaultdict

losses = defaultdict(list)

# Number of runs
for i in range(5):
    # get shuffled data
    train_loader, val_loader, test_loader = create_dataloader(
        batch_size=8, 
        test_batch_size=1,
        pin_memory=False,
        data_mask=[
            ("item", 1),
            ("store", 1)
        ],
        specs=all_specs,
        )

    for layer_combo in layer_combos:
        net_1 = simple_net(layer_combo)
        loss = partial(pinball_loss, h_cost=h_cost, l_cost=l_cost)
        optimizer = torch.optim.Adam(net_1.parameters(), lr=0.001)
        _, _ = train(net_1, optimizer, loss, train_loader, val_loader, epochs=200, eval_interval=10, device="cpu", use_tqdm=False)
        test_loss = get_test_loss(net_1, test_loader, loss, "cpu")
        
        losses[str(layer_combo)].append(torch.tensor(test_loss, dtype=torch.float32).mean().item())

        # Print average loss
        print(f"Test Loss for {str(layer_combo):<50}: {torch.tensor(test_loss, dtype=torch.float32).mean().item():.4f}")


Test Loss for [12, 30, 1]                                       : 4.3790
Test Loss for [12, 30, 30, 1]                                   : 4.0179
Test Loss for [12, 30, 30, 30, 1]                               : 4.0857
Test Loss for [12, 30, 30, 30, 30, 1]                           : 3.7098
Test Loss for [12, 30, 30, 30, 30, 30, 1]                       : 3.9422
Test Loss for [12, 40, 1]                                       : 3.7776
Test Loss for [12, 40, 40, 1]                                   : 4.0996
Test Loss for [12, 40, 40, 40, 1]                               : 3.7299
Test Loss for [12, 40, 40, 40, 40, 1]                           : 3.6025
Test Loss for [12, 40, 40, 40, 40, 40, 1]                       : 4.5201
Test Loss for [12, 50, 1]                                       : 4.2323
Test Loss for [12, 50, 50, 1]                                   : 4.1344
Test Loss for [12, 50, 50, 50, 50, 1]                           : 3.7320
Test Loss for [12, 50, 50, 50, 50, 50, 1]          

In [81]:
# Print the sorted lowest to highest average loss
for layer_combo, losses in sorted(losses.items(), key=lambda x: torch.tensor(x[1]).mean()):
    print(f"Layer Combo: {layer_combo}, Average Loss: {torch.tensor(losses).mean():.4f}")


Layer Combo: [12, 40, 40, 40, 1], Average Loss: 3.7477
Layer Combo: [12, 60, 60, 60, 1], Average Loss: 3.7549
Layer Combo: [12, 40, 40, 40, 40, 1], Average Loss: 3.8078
Layer Combo: [12, 60, 60, 60, 60, 60, 1], Average Loss: 3.8443
Layer Combo: [12, 40, 40, 1], Average Loss: 3.8640
Layer Combo: [12, 60, 60, 1], Average Loss: 3.8645
Layer Combo: [12, 30, 30, 30, 30, 1], Average Loss: 3.9003
Layer Combo: [12, 60, 60, 60, 60, 1], Average Loss: 3.9267
Layer Combo: [12, 40, 1], Average Loss: 3.9294
Layer Combo: [12, 30, 30, 30, 30, 30, 1], Average Loss: 3.9605
Layer Combo: [12, 30, 30, 30, 1], Average Loss: 3.9675
Layer Combo: [12, 30, 30, 1], Average Loss: 3.9700
Layer Combo: [12, 50, 50, 50, 50, 1], Average Loss: 4.0041
Layer Combo: [12, 40, 40, 40, 40, 40, 1], Average Loss: 4.0206
Layer Combo: [12, 60, 1], Average Loss: 4.0245
Layer Combo: [12, 50, 50, 50, 50, 50, 1], Average Loss: 4.0419
Layer Combo: [12, 50, 1], Average Loss: 4.0579
Layer Combo: [12, 50, 50, 1], Average Loss: 4.2098
La